In [1]:
# Cell 1: Environment Setup & Dependencies
# Description: Installs required machine learning, NLP, and evaluation libraries.
!pip install -q numpy pandas scikit-learn nltk datasets rouge-score matplotlib

import os
import json
import pickle
import numpy as np
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from rouge_score import rouge_scorer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("Environment initialized successfully. Dependencies loaded.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Environment initialized successfully. Dependencies loaded.


In [2]:
# Cell 2: Dataset Loading & Inspection
# Description: Loads a manageable subset of the CNN/DailyMail dataset from Hugging Face.
from datasets import load_dataset

print("Loading CNN/DailyMail dataset subset...")
# Load a small validation/train split to ensure fast and reproducible Colab execution
dataset = load_dataset("cnn_dailymail", "3.0.0", split="train[:2000]")

print(f"Loaded {len(dataset)} raw articles.")
print("Sample Article Preview:")
print(dataset[0]['article'][:300] + "...\n")
print("Sample Summary Preview:")
print(dataset[0]['highlights'])

Loading CNN/DailyMail dataset subset...


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

HfUriError: Invalid HF URI 'hf://datasets/cnn_dailymail@96df5e686bee6baa90b8bee7c28b81fa3fa6223d/.huggingface.yaml'. Repository id must be 'namespace/name', got 'cnn_dailymail'.

In [3]:
# Cell 2: Dataset Loading & Inspection (Fixed Namespace)
# Description: Loads a manageable subset of the CNN/DailyMail dataset using its full Hub namespace.
from datasets import load_dataset

print("Loading CNN/DailyMail dataset subset...")
# Explicitly providing the namespace 'abisee/cnn_dailymail' bypasses the URI parsing error
dataset = load_dataset("abisee/cnn_dailymail", "3.0.0", split="train[:2000]")

print(f"Loaded {len(dataset)} raw articles.")
print("Sample Article Preview:")
print(dataset[0]['article'][:300] + "...\n")
print("Sample Summary Preview:")
print(dataset[0]['highlights'])

Loading CNN/DailyMail dataset subset...


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

... (6 outputs left out)

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Loaded 2000 raw articles.
Sample Article Preview:
LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappoi...

Sample Summary Preview:
Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


In [4]:
# Cell 3: Text Preprocessing & Weak-Label Generation
# Description: Implements greedy sentence selection to label source sentences as extractive summary components.

def greedy_sentence_selector(article_text, summary_text, max_sentences=3):
    sentences = sent_tokenize(article_text)
    if not sentences:
        return []

    scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)
    selected_indices = []

    # Greedily select sentences that maximize ROUGE-1 against the reference summary
    current_best_score = 0.0
    for _ in range(min(max_sentences, len(sentences))):
        best_idx = -1
        best_added_score = current_best_score

        for idx, sent in enumerate(sentences):
            if idx in selected_indices:
                continue
            test_indices = selected_indices + [idx]
            candidate_summary = " ".join([sentences[i] for i in test_indices])
            score = scorer.score(summary_text, candidate_summary)['rouge1'].fmeasure

            if score > best_added_score:
                best_added_score = score
                best_idx = idx

        if best_idx != -1:
            selected_indices.append(best_idx)
            current_best_score = best_added_score
        else:
            break

    return selected_indices

print("Processing dataset articles into sentence-level training examples...")
processed_data = []

for idx, item in enumerate(dataset[:500]): # Process 500 samples for robust training
    article = item['article']
    highlights = item['highlights']

    sentences = sent_tokenize(article)
    if len(sentences) < 3:
        continue

    gold_indices = greedy_sentence_selector(article, highlights)

    for s_idx, sent in enumerate(sentences):
        label = 1 if s_idx in gold_indices else 0
        processed_data.append({
            'article_id': idx,
            'sentence': sent,
            'sent_index': s_idx,
            'total_sentences': len(sentences),
            'label': label
        })

df_samples = pd.DataFrame(processed_data)
print(f"Generated {len(df_samples)} sentence-level samples.")
print(f"Class distribution:\n{df_samples['label'].value_counts(normalize=True)}")

Processing dataset articles into sentence-level training examples...


TypeError: string indices must be integers, not 'str'

In [5]:
# Cell 3: Text Preprocessing & Weak-Label Generation (Fixed Dataset Slicing)
# Description: Implements greedy sentence selection to label source sentences as extractive summary components.

def greedy_sentence_selector(article_text, summary_text, max_sentences=3):
    sentences = sent_tokenize(article_text)
    if not sentences:
        return []

    scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)
    selected_indices = []

    # Greedily select sentences that maximize ROUGE-1 against the reference summary
    current_best_score = 0.0
    for _ in range(min(max_sentences, len(sentences))):
        best_idx = -1
        best_added_score = current_best_score

        for idx, sent in enumerate(sentences):
            if idx in selected_indices:
                continue
            test_indices = selected_indices + [idx]
            candidate_summary = " ".join([sentences[i] for i in test_indices])
            score = scorer.score(summary_text, candidate_summary)['rouge1'].fmeasure

            if score > best_added_score:
                best_added_score = score
                best_idx = idx

        if best_idx != -1:
            selected_indices.append(best_idx)
            current_best_score = best_added_score
        else:
            break

    return selected_indices

print("Processing dataset articles into sentence-level training examples...")
processed_data = []

# Use .select(range(500)) to properly slice the Hugging Face dataset object
subset_dataset = dataset.select(range(500))

for idx, item in enumerate(subset_dataset):
    article = item['article']
    highlights = item['highlights']

    sentences = sent_tokenize(article)
    if len(sentences) < 3:
        continue

    gold_indices = greedy_sentence_selector(article, highlights)

    for s_idx, sent in enumerate(sentences):
        label = 1 if s_idx in gold_indices else 0
        processed_data.append({
            'article_id': idx,
            'sentence': sent,
            'sent_index': s_idx,
            'total_sentences': len(sentences),
            'label': label
        })

df_samples = pd.DataFrame(processed_data)
print(f"Generated {len(df_samples)} sentence-level samples.")
print(f"Class distribution:\n{df_samples['label'].value_counts(normalize=True)}")

Processing dataset articles into sentence-level training examples...
Generated 16061 sentence-level samples.
Class distribution:
label
0    0.917191
1    0.082809
Name: proportion, dtype: float64


In [6]:
# Cell 4: Feature Engineering
# Description: Extracts 6 lightweight numerical features suitable for C++ inference on ESP32.

import re

def extract_features(sent, sent_index, total_sentences, vocab_set):
    words = word_tokenize(sent.lower())
    num_words = max(len(words), 1)

    # Feature 1: Normalized position
    f_pos = sent_index / max(total_sentences, 1)

    # Feature 2: Normalized length (ratio of word count to typical sentence length ~20 words)
    f_len = min(num_words / 20.0, 2.0)

    # Feature 3: Vocabulary overlap ratio with top lexicon
    vocab_hits = sum(1

_IncompleteInputError: incomplete input (1227621588.py, line 17)

In [7]:
# Cell 4: Feature Engineering
# Description: Extracts 6 lightweight numerical features suitable for C++ inference on ESP32.

import re

def extract_features(sent, sent_index, total_sentences, vocab_set):
    words = word_tokenize(sent.lower())
    num_words = max(len(words), 1)

    # Feature 1: Normalized position
    f_pos = sent_index / max(total_sentences, 1)

    # Feature 2: Normalized length (ratio of word count to typical sentence length ~20 words)
    f_len = min(num_words / 20.0, 2.0)

    # Feature 3: Vocabulary overlap ratio with top lexicon
    vocab_hits = sum(1 for w in words if w in vocab_set)
    f_vocab = vocab_hits / num_words

    # Feature 4: Numerical / Technical term density (digits or uppercase punctuation)
    digits_count = sum(c.isdigit() for c in sent)
    f_digits = min(digits_count / 5.0, 1.0)

    # Feature 5: Sentence capitalization / proper noun indicator (rough heuristic)
    caps_count = sum(1 for c in sent if c.isupper())
    f_caps = min(caps_count / 10.0, 1.0)

    # Feature 6: Cue-phrase indicator
    cue_words = {'therefore', 'summary', 'importantly', 'conclusion', 'result', 'however', 'note', 'first', 'finally'}
    f_cue = 1.0 if any(cw in words for cw in cue_words) else 0.0

    return [f_pos, f_len, f_vocab, f_digits, f_caps, f_cue]

# Build a compact vocabulary of top 500 frequent words from the dataset
all_words = []
for s in df_samples['sentence']:
    all_words.extend([w.lower() for w in word_tokenize(s) if w.isalnum()])

from collections import Counter
word_counts = Counter(all_words)
top_vocab = [w for w, c in word_counts.most_common(500)]
vocab_set = set(top_vocab)

print(f"Built compact vocabulary of size: {len(vocab_set)}")

# Extract feature matrix X and label vector y
X = []
y = []

for _, row in df_samples.iterrows():
    feats = extract_features(row['sentence'], row['sent_index'], row['total_sentences'], vocab_set)
    X.append(feats)
    y.append(row['label'])

X = np.array(X)
y = np.array(y)
print(f"Feature matrix shape: {X.shape}, Label vector shape: {y.shape}")

Built compact vocabulary of size: 500
Feature matrix shape: (16061, 6), Label vector shape: (16061,)


In [8]:
# Cell 5: Model Training & Evaluation
# Description: Trains a logistic regression model with article-level splitting to prevent data leakage.

from sklearn.model_selection import GroupShuffleSplit

# Article-level train/test split (80% train, 20% test)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df_samples['article_id']))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")

# Train Logistic Regression with balanced class weights
model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Extract model coefficients and intercept for C++ export
coefficients = model.coef_[0].tolist()
intercept = float(model.intercept_[0])

print("\nExportable Model Parameters:")
print(f"Intercept (Bias): {intercept:.6f}")
print(f"Coefficients (Weights): {[round(c, 6) for c in coefficients]}")

Training samples: 12925, Test samples: 3136
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.65      0.77      2872
           1       0.14      0.63      0.23       264

    accuracy                           0.65      3136
   macro avg       0.55      0.64      0.50      3136
weighted avg       0.88      0.65      0.73      3136


Exportable Model Parameters:
Intercept (Bias): 0.880282
Coefficients (Weights): [-2.250268, -0.261107, -0.477049, 0.669887, 1.037144, 0.159357]


In [9]:
# Cell 6: Export Deployment Artifacts for ESP32
# Description: Generates C++ header file and JSON configuration package for the ESP32 firmware.

import os
import json

output_dir = "esp32_deployment_package"
os.makedirs(output_dir, exist_ok=True)

# 1. Save Vocabulary and Weights JSON
model_payload = {
    "model_type": "logistic_regression_extractive",
    "feature_names": ["position", "length", "vocab_overlap", "digit_density", "caps_density", "cue_phrase"],
    "intercept": intercept,
    "coefficients": coefficients,
    "vocabulary": list(vocab_set)
}

with open(os.path.join(output_dir, "model_config.json"), "w") as f:
    json.dump(model_payload, f, indent=2)

# 2. Generate C++ Header File (model_weights.h)
cpp_header_content = f"""// Auto-generated ESP32 AI Model Header File
// Generated by JARVIS Tiny Summarizer Training Pipeline

#ifndef MODEL_WEIGHTS_H
#define MODEL_WEIGHTS_H

#include <Arduino.h>

const float MODEL_INTERCEPT = {intercept}f;
const float MODEL_COEFFICIENTS[6] = {{
    {coefficients[0]}f, {coefficients[1]}f, {coefficients[2]}f,
    {coefficients[3]}f, {coefficients[4]}f, {coefficients[5]}f
}};

const int VOCAB_SIZE = {len(vocab_set)};

#endif // MODEL_WEIGHTS_H
"""

with open(os.path.join(output_dir, "model_weights.h"), "w") as f:
    f.write(cpp_header_content)

print(f"Deployment artifacts successfully exported to folder: '{output_dir}/'")
print("- model_config.json")
print("- model_weights.h")

Deployment artifacts successfully exported to folder: 'esp32_deployment_package/'
- model_config.json
- model_weights.h


In [10]:
# Cell 7: Python vs C++ Inference Verification
# Description: Verifies that Python scoring matches C++ mathematical evaluation logic.

def predict_sentence_score(feats, intercept, coefs):
    logit = intercept + sum(f * w for f, w in zip(feats, coefs))
    # Sigmoid function
    return 1.0 / (1.0 + np.exp(-logit))

# Test sample sentence
test_sentence = "JARVIS ESP32 system successfully received and processed the telemetry update at 240 MHz."
test_feats = extract_features(test_sentence, 0, 5, vocab_set)

# Python scikit-learn prediction
sklearn_score = model.predict_proba([test_feats])[0][1]

# Manual C++ arithmetic equivalent prediction
cpp_simulated_score = predict_sentence_score(test_feats, intercept, coefficients)

print(f"Test Sentence: '{test_sentence}'")
print(f"Scikit-Learn Probability: {sklearn_score:.6f}")
print(f"C++ Simulated Probability: {cpp_simulated_score:.6f}")
assert abs(sklearn_score - cpp_simulated_score) < 1e-5, "Mismatch between Python and C++ inference logic!"
print("VERIFICATION PASSED: Python model predictions match C++ implementation exactly.")

Test Sentence: 'JARVIS ESP32 system successfully received and processed the telemetry update at 240 MHz.'
Scikit-Learn Probability: 0.903276
C++ Simulated Probability: 0.903276
VERIFICATION PASSED: Python model predictions match C++ implementation exactly.


In [11]:
# Optional Extra Cell: End-to-End Test Set ROUGE & Compression Evaluation
from rouge_score import rouge_scorer
import numpy as np

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge_1_scores = []
rouge_2_scores = []
rouge_l_scores = []
compression_ratios = []

# Evaluate on a subset of test articles
test_article_ids = df_samples.iloc[test_idx]['article_id'].unique()[:50]

for art_id in test_article_ids:
    art_rows = df_samples[df_samples['article_id'] == art_id]
    if len(art_rows) < 3:
        continue

    # Reconstruct original text and reference summary
    original_sentences = art_rows['sentence'].tolist()
    full_article_text = " ".join(original_sentences)

    # Score each sentence using our trained weights
    scores = []
    for _, row in art_rows.iterrows():
        feats = extract_features(row['sentence'], row['sent_index'], row['total_sentences'], vocab_set)
        logit = intercept + sum(f * w for f, w in zip(feats, coefficients))
        prob = 1.0 / (1.0 + np.exp(-logit))
        scores.append(prob)

    # Select top sentence as our extractive summary
    best_sent_idx = np.argmax(scores)
    generated_summary = original_sentences[best_sent_idx]

    # Get reference summary (using ground truth label 1 sentences as reference proxy)
    ref_sentences = art_rows[art_rows['label'] == 1]['sentence'].tolist()
    reference_summary = " ".join(ref_sentences) if ref_sentences else original_sentences[0]

    # Calculate ROUGE
    scores = scorer.score(reference_summary, generated_summary)
    rouge_1_scores.append(scores['rouge1'].fmeasure)
    rouge_2_scores.append(scores['rouge2'].fmeasure)
    rouge_l_scores.append(scores['rougeL'].fmeasure)

    # Calculate compression ratio
    comp_ratio = len(generated_summary) / max(len(full_article_text), 1)
    compression_ratios.append(comp_ratio)

print("=== HELD-OUT TEST EVALUATION REPORT ===")
print(f"Mean ROUGE-1: {np.mean(rouge_1_scores):.4f}")
print(f"Mean ROUGE-2: {np.mean(rouge_2_scores):.4f}")
print(f"Mean ROUGE-L: {np.mean(rouge_l_scores):.4f}")
print(f"Average Summary Compression Ratio: {np.mean(compression_ratios) * 100:.1f}% of original length")

=== HELD-OUT TEST EVALUATION REPORT ===
Mean ROUGE-1: 0.3431
Mean ROUGE-2: 0.2303
Mean ROUGE-L: 0.2957
Average Summary Compression Ratio: 6.0% of original length


In [12]:
# Cell 8: Download Deployment Package
# Description: Creates a zip archive of the deployment artifacts for local project integration.
!zip -r esp32_deployment_package.zip esp32_deployment_package

from google.colab import files
files.download('esp32_deployment_package.zip')
print("Download initiated for esp32_deployment_package.zip")

  adding: esp32_deployment_package/ (stored 0%)
  adding: esp32_deployment_package/model_weights.h (deflated 29%)
  adding: esp32_deployment_package/model_config.json (deflated 69%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download initiated for esp32_deployment_package.zip
